# Section 1: Lookups & Joins (Q01–Q10)
Pure SQL queries using ontology metadata. All relationships use `direct_join`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))
from pcg_example.benchmark.notebook_helpers import get_session, run_sql
conn, ontology = get_session()

## Q01 — Supplier lookup
Pull the full supplier profile for supplier code SUP-012 — name, city, country, tier, and active status.

In [ ]:
# IDENTIFY: Supplier entity | DISPATCH: direct_join (SQL)
# PARAMETERIZE: table=suppliers, identifier=supplier_code
run_sql(conn, """
    SELECT supplier_code, name, city, country, tier, is_active
    FROM suppliers
    WHERE supplier_code = 'SUP-012'
""")

## Q02 — Supplier ingredients with -ALT check
Which ingredients does Apex Ingredients LLC (SUP-003) offer, and at what unit cost? Include any alternate supplier records that might be duplicates — we've seen "-ALT" codes creep into the master data (check for SUP-003-ALT).

In [ ]:
# IDENTIFY: Supplier -> SupplierOffersIngredient -> Ingredient
# DISPATCH: direct_join | Edge attributes: unit_cost
run_sql(conn, """
    SELECT s.supplier_code, s.name as supplier_name,
           i.ingredient_code, i.name as ingredient_name,
           si.unit_cost, si.lead_time_days, si.min_order_qty
    FROM suppliers s
    JOIN supplier_ingredients si ON si.supplier_id = s.id
    JOIN ingredients i ON si.ingredient_id = i.id
    WHERE s.supplier_code IN ('SUP-003', 'SUP-003-ALT')
    ORDER BY s.supplier_code, i.ingredient_code
""")

## Q03 — Open PO exposure by plant
What is our total open purchase order exposure by plant? Show only POs with status "open" and break out the total line-item value for Dallas (PLANT-TX), Columbus (PLANT-OH), Sacramento (PLANT-CA), and Atlanta (PLANT-GA).

In [ ]:
# IDENTIFY: PurchaseOrder -> POHasLines -> PurchaseOrderLine, POAtPlant -> Plant
# DISPATCH: direct_join | Filter: status='open'
run_sql(conn, """
    SELECT p.plant_code, p.name as plant_name,
           COUNT(DISTINCT po.id) as open_po_count,
           SUM(pol.quantity_kg * pol.unit_cost) as total_line_value
    FROM purchase_orders po
    JOIN purchase_order_lines pol ON pol.po_id = po.id
    JOIN plants p ON po.plant_id = p.id
    WHERE po.status = 'open'
      AND p.plant_code IN ('PLANT-TX', 'PLANT-OH', 'PLANT-CA', 'PLANT-GA')
    GROUP BY p.plant_code, p.name
    ORDER BY total_line_value DESC
""")

## Q04 — Premium Oral Care SKUs above $45
List all Oral Care SKUs in the Premium segment with a price per case above $45. Include the brand and whether the SKU is currently active.

In [ ]:
# IDENTIFY: SKU entity | DISPATCH: direct_join
# Note: column is value_segment (not segment)
run_sql(conn, """
    SELECT sku_code, name, brand, price_per_case, is_active
    FROM skus
    WHERE category = 'Oral Care'
      AND value_segment = 'Premium'
      AND price_per_case > 45
    ORDER BY price_per_case DESC
""")

## Q05 — Order lines for specific order
Show me every line on order ORD-1-GRO-DC-001-28 — SKU, quantity in cases, unit price, and line status.

In [ ]:
# IDENTIFY: Order -> OrderHasLines -> OrderLine -> OrderLineForSKU -> SKU
# DISPATCH: direct_join | Composite PK on order_lines
run_sql(conn, """
    SELECT ol.line_number, s.sku_code, s.name as sku_name,
           ol.quantity_cases, ol.unit_price, ol.status
    FROM orders o
    JOIN order_lines ol ON ol.order_id = o.id
    JOIN skus s ON ol.sku_id = s.id
    WHERE o.order_number = 'ORD-1-GRO-DC-001-28'
    ORDER BY ol.line_number
""")

## Q06 — Suppliers for formula ingredients
For formula FORM-BULK-OC-ORIGINAL-027, which suppliers can provide the required ingredients? Show the ingredient name, supplier name, and the supplier's quoted unit cost.

In [ ]:
# IDENTIFY: Formula -> FormulaHasIngredients -> Ingredient -> SupplierOffersIngredient -> Supplier
# DISPATCH: direct_join chain
run_sql(conn, """
    SELECT i.ingredient_code, i.name as ingredient_name,
           s.supplier_code, s.name as supplier_name,
           si.unit_cost
    FROM formulas f
    JOIN formula_ingredients fi ON fi.formula_id = f.id
    JOIN ingredients i ON fi.ingredient_id = i.id
    JOIN supplier_ingredients si ON si.ingredient_id = i.id
    JOIN suppliers s ON si.supplier_id = s.id
    WHERE f.formula_code = 'FORM-BULK-OC-ORIGINAL-027'
    ORDER BY i.ingredient_code, si.unit_cost
""")

## Q07 — Revenue by channel
Break down our total revenue by channel for the full year. Rank channels from highest to lowest and include the order count behind each.

> **Note:** Revenue is in `ar_invoices` (NOT `order_lines`, which has `unit_price` always zero).

In [ ]:
# IDENTIFY: ARInvoice (has channel column) | DISPATCH: direct_join
# INTERPRET: ar_invoices.channel is the source of truth for revenue by channel
run_sql(conn, """
    SELECT ari.channel,
           COUNT(DISTINCT ari.id) as invoice_count,
           SUM(ari.total_amount) as total_revenue
    FROM ar_invoices ari
    GROUP BY ari.channel
    ORDER BY total_revenue DESC
""")

## Q08 — Orders by status between day 90-120
How many orders were placed between day 90 and day 120? Summarize by status — pending, allocated, shipped, delivered — so I can see the pipeline.

In [ ]:
# IDENTIFY: Order entity | DISPATCH: direct_join
# Note: dates are integer day numbers (1-365)
run_sql(conn, """
    SELECT status, COUNT(*) as order_count
    FROM orders
    WHERE day BETWEEN 90 AND 120
    GROUP BY status
    ORDER BY order_count DESC
""")

## Q09 — Consolidated location list
Give me a single consolidated list of every location in our network — plants, distribution centers, and retail locations — with city, country, and location type. I need the full picture.

In [ ]:
# IDENTIFY: Plant + DistributionCenter + RetailLocation (polymorphic Location)
# DISPATCH: UNION ALL across location tables
run_sql(conn, """
    SELECT plant_code as location_code, name, city, country, 'plant' as location_type
    FROM plants
    UNION ALL
    SELECT dc_code, name, city, country, type as location_type
    FROM distribution_centers
    UNION ALL
    SELECT location_code, name, city, country, 'store' as location_type
    FROM retail_locations
    ORDER BY location_type, location_code
""")

## Q10 — Long lead time supplier-ingredient combos
Which supplier-ingredient combinations have a lead time greater than 30 days? Include the unit cost and minimum order quantity — I want to see our long-tail procurement exposure.

In [ ]:
# IDENTIFY: SupplierOffersIngredient (edge attributes: lead_time_days, unit_cost, min_order_qty)
# DISPATCH: direct_join
run_sql(conn, """
    SELECT s.supplier_code, s.name as supplier_name,
           i.ingredient_code, i.name as ingredient_name,
           si.lead_time_days, si.unit_cost, si.min_order_qty
    FROM supplier_ingredients si
    JOIN suppliers s ON si.supplier_id = s.id
    JOIN ingredients i ON si.ingredient_id = i.id
    WHERE si.lead_time_days > 30
    ORDER BY si.lead_time_days DESC
""")

In [ ]:
conn.close()
print("Session closed.")